<a href="https://colab.research.google.com/github/chawlaniharika23/Fraud_Detection_Research/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01 — Exploratory Data Analysis (EDA)

## Financial Fraud Detection: PaySim and IEEE-CIS

**Purpose:** Understand the structure, quality, class distribution, transaction behaviour, and temporal characteristics of the two datasets before preprocessing and model development.

### Research workflow
1. Inspect both datasets and their schemas.
2. Quantify class imbalance and missingness.
3. Explore transaction amount, type, balances, and time-related patterns.
4. Record modelling implications without fitting or selecting a predictive model.

### Important experimental safeguards
- This notebook is exploratory; it does **not** train a model or select features using the test set.
- `isFraud` is the target label, not an input feature.
- IEEE-CIS `test_transaction.csv` has no public labels and is not used to calculate fraud rates.
- PaySim balance-error fields are diagnostic EDA variables. Their availability at prediction time must be assessed separately before they are used for real-time modelling.
- All plots are descriptive. Correlation or association does not establish causation.


## 1. Environment and imports

In [ ]:
import os
import sys
import gc
import shutil
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Seaborn:", sns.__version__)
print("Environment initialized.")

## 2. Mount Google Drive and configure paths

The original EDA notebook used `fraud-detection-data`, while Notebook 02 expects `fraud-detection-research/data`. This version resolves that inconsistency.

- `DATA_DIR` is the canonical raw-data directory used by the rest of the project.
- `ARCHIVE_DIR` is the original archive location. If your ZIP files are elsewhere, change only this variable.
- If CSVs are already extracted, the notebook uses them and does not extract the archives again.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive")

# Canonical location expected by Notebook 02 and subsequent notebooks.
DATA_DIR = DRIVE_ROOT / "fraud-detection-research" / "data"

# Original archive location used in the earlier version of Notebook 01.
ARCHIVE_DIR = DRIVE_ROOT / "fraud-detection-data"

PAYSIM_ARCHIVE = ARCHIVE_DIR / "archive.zip"
IEEE_ARCHIVE = ARCHIVE_DIR / "ieee-fraud-detection.zip"

PAYSIM_FILENAME = "PS_20174392719_1491204439457_log.csv"
IEEE_TRANSACTION_FILENAME = "train_transaction.csv"
IEEE_IDENTITY_FILENAME = "train_identity.csv"

PAYSIM_EXTRACT_DIR = DATA_DIR / "paysim"
IEEE_EXTRACT_DIR = DATA_DIR / "ieee"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PAYSIM_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
IEEE_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

print("Canonical data directory:", DATA_DIR)
print("Archive directory:", ARCHIVE_DIR)
print("Existing files in canonical directory:")
for item in sorted(DATA_DIR.iterdir()):
    print(" -", item.name)

## 3. Locate and extract source datasets

In [ ]:
def find_file_recursively(root: Path, filename: str):
    """Return the first exact filename match below root, or None."""
    matches = list(root.rglob(filename))
    return matches[0] if matches else None


def safe_extract_zip(zip_path: Path, destination: Path):
    """Extract a ZIP archive while preventing path traversal."""
    if not zip_path.exists():
        raise FileNotFoundError(
            f"Archive not found: {zip_path}\n"
            "Update ARCHIVE_DIR in Section 2 or place the ZIP file there."
        )

    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as archive:
        root = destination.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f"Unsafe path in archive: {member.filename}")
        archive.extractall(destination)

    print(f"Extracted {zip_path.name} into {destination}")


# Reuse extracted data whenever possible.
paysim_source = find_file_recursively(DATA_DIR, PAYSIM_FILENAME)
ieee_transaction_source = find_file_recursively(DATA_DIR, IEEE_TRANSACTION_FILENAME)
ieee_identity_source = find_file_recursively(DATA_DIR, IEEE_IDENTITY_FILENAME)

if paysim_source is None:
    safe_extract_zip(PAYSIM_ARCHIVE, PAYSIM_EXTRACT_DIR)
    paysim_source = find_file_recursively(PAYSIM_EXTRACT_DIR, PAYSIM_FILENAME)

if ieee_transaction_source is None:
    safe_extract_zip(IEEE_ARCHIVE, IEEE_EXTRACT_DIR)
    ieee_transaction_source = find_file_recursively(IEEE_EXTRACT_DIR, IEEE_TRANSACTION_FILENAME)
    ieee_identity_source = find_file_recursively(IEEE_EXTRACT_DIR, IEEE_IDENTITY_FILENAME)

if paysim_source is None:
    raise FileNotFoundError(f"Could not find {PAYSIM_FILENAME} after extraction.")
if ieee_transaction_source is None:
    raise FileNotFoundError(f"Could not find {IEEE_TRANSACTION_FILENAME} after extraction.")

# Copy the source CSVs to the canonical root because Notebook 02 reads them there.
for source, filename in [
    (paysim_source, PAYSIM_FILENAME),
    (ieee_transaction_source, IEEE_TRANSACTION_FILENAME),
    (ieee_identity_source, IEEE_IDENTITY_FILENAME),
]:
    if source is not None:
        destination = DATA_DIR / filename
        if source.resolve() != destination.resolve():
            shutil.copy2(source, destination)
        print(f"Ready: {destination.name} ({destination.stat().st_size / (1024**2):.1f} MB)")

PAYSIM_FILE = DATA_DIR / PAYSIM_FILENAME
IEEE_TRANSACTION_FILE = DATA_DIR / IEEE_TRANSACTION_FILENAME
IEEE_IDENTITY_FILE = DATA_DIR / IEEE_IDENTITY_FILENAME

## 4. Dataset inventory and previews

In [ ]:
def inspect_csv_file(path: Path, nrows=3):
    sample = pd.read_csv(path, nrows=nrows, low_memory=False)
    print(f"File: {path.name}")
    print(f"Sample shape: {sample.shape}")
    print("Columns:", sample.columns.tolist()[:20])
    display(sample.head(nrows))


for path in [PAYSIM_FILE, IEEE_TRANSACTION_FILE]:
    inspect_csv_file(path)

if IEEE_IDENTITY_FILE.exists():
    inspect_csv_file(IEEE_IDENTITY_FILE)
else:
    print("IEEE identity file not found; EDA will proceed with transaction data only.")

## 5. Load the labelled datasets

PaySim and IEEE-CIS are loaded separately because their schemas are different. The IEEE-CIS transaction training file contains the `isFraud` target. The public test transaction file is intentionally excluded from labelled EDA.

In [ ]:
paysim_df = pd.read_csv(PAYSIM_FILE, low_memory=False)
ieee_train_df = pd.read_csv(IEEE_TRANSACTION_FILE, low_memory=False)

print("PaySim shape:", paysim_df.shape)
print("IEEE-CIS train transaction shape:", ieee_train_df.shape)

required_paysim = {"isFraud", "step", "type", "amount"}
required_ieee = {"isFraud", "TransactionID", "TransactionDT", "TransactionAmt"}

missing_paysim = required_paysim - set(paysim_df.columns)
missing_ieee = required_ieee - set(ieee_train_df.columns)

if missing_paysim:
    raise ValueError(f"PaySim is missing required columns: {missing_paysim}")
if missing_ieee:
    raise ValueError(f"IEEE-CIS is missing required columns: {missing_ieee}")

print("Required target and analysis columns are present.")
display(paysim_df.head())
display(ieee_train_df.head())

## 6. Dataset structure and data types

In [ ]:
def dataset_profile(df, name):
    profile = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(dropna=True)
    })
    print(f"\n{name}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    print("Duplicate rows:", f"{df.duplicated().sum():,}")
    print("Memory usage (MB):", round(df.memory_usage(deep=True).sum() / 1024**2, 2))
    display(profile.sort_values("missing_pct", ascending=False))
    return profile

paysim_profile = dataset_profile(paysim_df, "PaySim")
ieee_profile = dataset_profile(ieee_train_df, "IEEE-CIS training transactions")

## 7. Target distribution and class imbalance

In [ ]:
def target_summary(df, dataset_name, target="isFraud"):
    counts = df[target].value_counts(dropna=False).sort_index()
    percentages = df[target].value_counts(normalize=True, dropna=False).sort_index() * 100

    summary = pd.DataFrame({
        "Transaction_Count": counts,
        "Percentage": percentages
    })
    summary.index = summary.index.map({0: "Legitimate (0)", 1: "Fraudulent (1)"})
    print(f"\n{dataset_name} — class distribution")
    display(summary.round(4))

    if 1 in df[target].values and 0 in df[target].values:
        fraud_rate = (df[target] == 1).mean() * 100
        imbalance_ratio = (df[target] == 0).sum() / max((df[target] == 1).sum(), 1)
        print(f"Fraud prevalence: {fraud_rate:.4f}%")
        print(f"Legitimate-to-fraud ratio: {imbalance_ratio:,.1f}:1")
    return summary

paysim_target_summary = target_summary(paysim_df, "PaySim")
ieee_target_summary = target_summary(ieee_train_df, "IEEE-CIS")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, df, name in [
    (axes[0, 0], paysim_df, "PaySim — counts"),
    (axes[0, 1], ieee_train_df, "IEEE-CIS — counts")
]:
    counts = df["isFraud"].value_counts().reindex([0, 1], fill_value=0)
    sns.barplot(x=["Legitimate", "Fraud"], y=counts.values, ax=ax)
    ax.set_title(name)
    ax.set_ylabel("Transactions")
    ax.ticklabel_format(axis="y", style="plain")

for ax, df, name in [
    (axes[1, 0], paysim_df, "PaySim — log counts"),
    (axes[1, 1], ieee_train_df, "IEEE-CIS — log counts")
]:
    counts = df["isFraud"].value_counts().reindex([0, 1], fill_value=0)
    sns.barplot(x=["Legitimate", "Fraud"], y=counts.values, ax=ax)
    ax.set_yscale("log")
    ax.set_title(name)
    ax.set_ylabel("Transactions (log scale)")

plt.suptitle("Class imbalance in the labelled datasets", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

### Interpretation
Fraud detection is an imbalanced classification problem. Accuracy alone can be misleading because a classifier that predicts every transaction as legitimate may still achieve a high accuracy. Later model comparisons must include precision, recall, F1-score, PR-AUC, confusion matrices, and false-positive rates.

## 8. Missing-value analysis

In [ ]:
def missing_value_report(df, dataset_name):
    report = pd.DataFrame({
        "Missing_Count": df.isna().sum(),
        "Missing_Percentage": df.isna().mean() * 100
    }).sort_values("Missing_Percentage", ascending=False)

    print(f"{dataset_name}: total missing cells = {int(df.isna().sum().sum()):,}")
    print("Columns with at least one missing value:",
          int((report["Missing_Count"] > 0).sum()))
    display(report[report["Missing_Count"] > 0].head(30))
    return report

paysim_missing_summary = missing_value_report(paysim_df, "PaySim")
ieee_missing_summary = missing_value_report(ieee_train_df, "IEEE-CIS")

In [ ]:
top_n = 25
top_missing = ieee_missing_summary.head(top_n).sort_values("Missing_Percentage")

plt.figure(figsize=(12, 8))
plt.barh(top_missing.index, top_missing["Missing_Percentage"])
plt.xlabel("Missing values (%)")
plt.ylabel("Feature")
plt.title(f"IEEE-CIS: top {top_n} features by missingness")
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

print("IEEE-CIS columns with >90% missing:",
      int((ieee_missing_summary["Missing_Percentage"] > 90).sum()))
print("IEEE-CIS columns with >50% missing:",
      int((ieee_missing_summary["Missing_Percentage"] > 50).sum()))

### Interpretation
IEEE-CIS contains substantial missingness across many columns. Missingness itself may carry information, but imputation and any missingness indicators must be fitted or decided using training data only in Notebook 02. PaySim's missingness should be verified from the actual loaded file rather than assumed.

## 9. PaySim: transaction type and fraud patterns

In [ ]:
fraud_by_type = (
    paysim_df.groupby("type", dropna=False)["isFraud"]
    .agg(Total_Transactions="count", Fraud_Cases="sum", Fraud_Rate="mean")
)
fraud_by_type["Fraud_Rate_Percent"] = fraud_by_type["Fraud_Rate"] * 100
fraud_by_type = fraud_by_type.sort_values("Fraud_Rate_Percent", ascending=False)

display(fraud_by_type.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_order = fraud_by_type.index.tolist()
sns.barplot(
    x=type_order,
    y=fraud_by_type.loc[type_order, "Total_Transactions"],
    ax=axes[0]
)
axes[0].set_title("PaySim transaction volume by type")
axes[0].set_xlabel("Transaction type")
axes[0].set_ylabel("Number of transactions")
axes[0].tick_params(axis="x", rotation=30)
axes[0].ticklabel_format(axis="y", style="plain")

sns.barplot(
    x=type_order,
    y=fraud_by_type.loc[type_order, "Fraud_Rate_Percent"],
    ax=axes[1]
)
axes[1].set_title("PaySim fraud rate by transaction type")
axes[1].set_xlabel("Transaction type")
axes[1].set_ylabel("Fraud rate (%)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
print("Fraud cases by transaction type:")
display(
    paysim_df.loc[paysim_df["isFraud"] == 1, "type"]
    .value_counts(dropna=False)
    .rename("Fraud_Cases")
    .to_frame()
)

### Interpretation
Transaction type can be associated with different fraud rates. Both the absolute fraud count and the within-type fraud rate matter: a type can have a high rate but a small transaction volume. This association is descriptive and should be validated on held-out data during modelling.

## 10. PaySim: transaction amount analysis

In [ ]:
amount_summary = paysim_df.groupby("isFraud")["amount"].agg(
    Transaction_Count="count",
    Mean="mean",
    Median="median",
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
    Minimum="min",
    Maximum="max"
)
amount_summary.index = amount_summary.index.map({0: "Legitimate", 1: "Fraudulent"})
display(amount_summary.round(2))

In [ ]:
# Use log1p for visualization because transaction amounts are highly skewed.
amount_plot = paysim_df[["isFraud", "amount"]].copy()
amount_plot["log1p_amount"] = np.log1p(amount_plot["amount"].clip(lower=0))

# Sample only for plotting; summary statistics above use the full dataset.
plot_parts = []
for label in [0, 1]:
    group = amount_plot[amount_plot["isFraud"] == label]
    n = min(100_000, len(group))
    if n:
        plot_parts.append(group.sample(n=n, random_state=RANDOM_STATE))
amount_plot_sample = pd.concat(plot_parts, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=amount_plot_sample, x="isFraud", y="log1p_amount",
            showfliers=False, ax=axes[0])
axes[0].set_title("PaySim transaction amount (log1p)")
axes[0].set_xlabel("0 = legitimate, 1 = fraud")
axes[0].set_ylabel("log(1 + amount)")

sns.violinplot(data=amount_plot_sample, x="isFraud", y="log1p_amount",
               inner="quartile", cut=0, ax=axes[1])
axes[1].set_title("PaySim amount distribution (sampled)")
axes[1].set_xlabel("0 = legitimate, 1 = fraud")
axes[1].set_ylabel("log(1 + amount)")

plt.tight_layout()
plt.show()

## 11. PaySim: balance consistency diagnostics

In [ ]:
# These are exploratory diagnostics, not automatically approved model features.
paysim_df["origin_balance_error_eda"] = (
    paysim_df["oldbalanceOrg"] - paysim_df["amount"] - paysim_df["newbalanceOrig"]
)
paysim_df["destination_balance_error_eda"] = (
    paysim_df["oldbalanceDest"] + paysim_df["amount"] - paysim_df["newbalanceDest"]
)

balance_summary = paysim_df.groupby("isFraud")[
    ["origin_balance_error_eda", "destination_balance_error_eda"]
].agg(["mean", "median", "std", "min", "max"])

display(balance_summary.round(2))

In [ ]:
# Robust plotting: sample each class and suppress extreme fliers for readability.
balance_cols = ["origin_balance_error_eda", "destination_balance_error_eda"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, balance_cols):
    parts = []
    for label in [0, 1]:
        group = paysim_df.loc[paysim_df["isFraud"] == label, ["isFraud", col]]
        n = min(100_000, len(group))
        if n:
            parts.append(group.sample(n=n, random_state=RANDOM_STATE))
    plot_df = pd.concat(parts, ignore_index=True)

    sns.boxplot(data=plot_df, x="isFraud", y=col, showfliers=False, ax=ax)
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_title(col.replace("_", " ").title())
    ax.set_xlabel("0 = legitimate, 1 = fraud")
    ax.set_ylabel("Balance error")

plt.tight_layout()
plt.show()

### Interpretation and modelling caution
Balance discrepancies can separate transaction groups in a simulated dataset. However, `newbalanceOrig` and `newbalanceDest` are post-transaction values. A model intended to score a transaction before it is completed must not use information that is only available afterward. Notebook 02 should therefore keep real-time and extended feature sets distinct and document the intended prediction point.

## 12. PaySim: temporal behaviour

In [ ]:
time_summary = (
    paysim_df.groupby("step")["isFraud"]
    .agg(total_transactions="size", fraud_cases="sum")
)
time_summary["fraud_rate_percent"] = (
    time_summary["fraud_cases"] / time_summary["total_transactions"] * 100
)

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

axes[0].plot(time_summary.index, time_summary["total_transactions"], linewidth=1)
axes[0].set_title("PaySim transaction volume by simulation step")
axes[0].set_ylabel("Transactions")

axes[1].plot(time_summary.index, time_summary["fraud_cases"], linewidth=1)
axes[1].set_title("PaySim fraud count by simulation step")
axes[1].set_ylabel("Fraud cases")

axes[2].plot(time_summary.index, time_summary["fraud_rate_percent"], linewidth=1)
axes[2].set_title("PaySim fraud rate by simulation step")
axes[2].set_xlabel("Simulation step")
axes[2].set_ylabel("Fraud rate (%)")

plt.tight_layout()
plt.show()

print("Unique simulation steps:", paysim_df["step"].nunique())
print("Step range:", paysim_df["step"].min(), "to", paysim_df["step"].max())

In [ ]:
# Avoid ranking steps with very few transactions as if their rates were equally reliable.
MIN_TRANSACTIONS_PER_STEP = 100
eligible_steps = time_summary[
    time_summary["total_transactions"] >= MIN_TRANSACTIONS_PER_STEP
]

print(f"Steps with at least {MIN_TRANSACTIONS_PER_STEP} transactions:",
      len(eligible_steps))
print("Highest fraud-rate steps among eligible steps:")
display(
    eligible_steps.sort_values("fraud_rate_percent", ascending=False).head(10)
)

## 13. IEEE-CIS: transaction amount and time patterns

In [ ]:
ieee_amount_summary = ieee_train_df.groupby("isFraud")["TransactionAmt"].agg(
    Transaction_Count="count",
    Mean="mean",
    Median="median",
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
    Maximum="max"
)
ieee_amount_summary.index = ieee_amount_summary.index.map(
    {0: "Legitimate", 1: "Fraudulent"}
)
display(ieee_amount_summary.round(2))

In [ ]:
ieee_plot = ieee_train_df[["isFraud", "TransactionAmt", "TransactionDT"]].copy()
ieee_plot["log1p_amount"] = np.log1p(ieee_plot["TransactionAmt"].clip(lower=0))

plot_parts = []
for label in [0, 1]:
    group = ieee_plot[ieee_plot["isFraud"] == label]
    n = min(100_000, len(group))
    if n:
        plot_parts.append(group.sample(n=n, random_state=RANDOM_STATE))
ieee_amount_sample = pd.concat(plot_parts, ignore_index=True)

plt.figure(figsize=(9, 5))
sns.boxplot(data=ieee_amount_sample, x="isFraud", y="log1p_amount",
            showfliers=False)
plt.title("IEEE-CIS transaction amount by fraud status (log1p, sampled)")
plt.xlabel("0 = legitimate, 1 = fraud")
plt.ylabel("log(1 + TransactionAmt)")
plt.tight_layout()
plt.show()

In [ ]:
# TransactionDT is a relative time variable, not a calendar timestamp.
# Bin it into equal-width intervals to inspect broad temporal variation.
ieee_time = ieee_train_df[["TransactionDT", "isFraud"]].dropna().copy()
ieee_time["time_bin"] = pd.cut(ieee_time["TransactionDT"], bins=20, duplicates="drop")

ieee_time_summary = (
    ieee_time.groupby("time_bin", observed=False)["isFraud"]
    .agg(total_transactions="size", fraud_cases="sum")
)
ieee_time_summary["fraud_rate_percent"] = (
    ieee_time_summary["fraud_cases"] /
    ieee_time_summary["total_transactions"].replace(0, np.nan) * 100
)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
x = np.arange(len(ieee_time_summary))
axes[0].plot(x, ieee_time_summary["total_transactions"], marker="o")
axes[0].set_title("IEEE-CIS transaction volume across relative-time bins")
axes[0].set_ylabel("Transactions")
axes[1].plot(x, ieee_time_summary["fraud_rate_percent"], marker="o")
axes[1].set_title("IEEE-CIS fraud rate across relative-time bins")
axes[1].set_xlabel("Ordered relative-time bin")
axes[1].set_ylabel("Fraud rate (%)")
plt.tight_layout()
plt.show()

display(ieee_time_summary.head(20).round(3))

### Interpretation
IEEE-CIS `TransactionDT` is a relative time offset, not a directly interpretable calendar date. The binned analysis is intended to reveal broad temporal variation only. Any temporal split or time-derived feature engineering should be performed consistently in Notebook 02.

## 14. Categorical and numerical feature overview

In [ ]:
def feature_type_summary(df, dataset_name, target="isFraud"):
    feature_df = df.drop(columns=[target], errors="ignore")
    numeric_cols = feature_df.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = feature_df.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    print(f"{dataset_name}")
    print("Numeric features:", len(numeric_cols))
    print("Categorical/text features:", len(categorical_cols))
    print("\nCategorical columns (first 40):")
    print(categorical_cols[:40])
    print("\nHigh-cardinality columns (top 20):")
    cardinality = feature_df.nunique(dropna=True).sort_values(ascending=False)
    display(cardinality.head(20).to_frame("Unique_Values"))
    return numeric_cols, categorical_cols

paysim_numeric_cols, paysim_categorical_cols = feature_type_summary(
    paysim_df, "PaySim"
)
ieee_numeric_cols, ieee_categorical_cols = feature_type_summary(
    ieee_train_df, "IEEE-CIS"
)

## 15. EDA findings and implications for modelling

In [ ]:
def fraud_rate_percent(df, target="isFraud"):
    return float(df[target].mean() * 100)

eda_summary = pd.DataFrame([
    {
        "Dataset": "PaySim",
        "Rows": len(paysim_df),
        "Features_including_target": paysim_df.shape[1],
        "Fraud_Count": int(paysim_df["isFraud"].sum()),
        "Fraud_Rate_Percent": fraud_rate_percent(paysim_df),
        "Missing_Cells": int(paysim_df.isna().sum().sum()),
        "Duplicate_Rows": int(paysim_df.duplicated().sum())
    },
    {
        "Dataset": "IEEE-CIS train transactions",
        "Rows": len(ieee_train_df),
        "Features_including_target": ieee_train_df.shape[1],
        "Fraud_Count": int(ieee_train_df["isFraud"].sum()),
        "Fraud_Rate_Percent": fraud_rate_percent(ieee_train_df),
        "Missing_Cells": int(ieee_train_df.isna().sum().sum()),
        "Duplicate_Rows": int(ieee_train_df.duplicated().sum())
    }
])

display(eda_summary.round(4))

## Key findings from the exploratory analysis

Use the actual tables and plots generated above when writing the final report; do not assume the same counts or percentages if the source files differ.

1. **Class imbalance:** both datasets contain far fewer fraudulent transactions than legitimate ones. Accuracy must not be the only model-selection metric.
2. **Missing data:** IEEE-CIS has substantial missingness. Missing-value handling must be learned from training data and applied consistently to validation and test data.
3. **PaySim transaction type:** fraud prevalence differs by transaction type. Both fraud count and fraud rate should be considered.
4. **Transaction amounts:** amount distributions are skewed, so log transformations are useful for visualization and may be worth testing as model features.
5. **Balance diagnostics:** origin and destination balance errors can reveal inconsistencies, but post-transaction balances may not be available at the time a real-time decision is made.
6. **Temporal variation:** fraud rates and transaction volumes vary over time. Temporal validation is important for evaluating generalization and potential distribution shift.
7. **Dataset differences:** PaySim is simulated mobile-money data, while IEEE-CIS is a separate transaction-fraud dataset. Their features and label distributions are not interchangeable.

### Handoff to Notebook 02
Notebook 02 should:
- Load raw files from `DATA_DIR` defined here (`/content/drive/MyDrive/fraud-detection-research/data`).
- Keep PaySim and IEEE-CIS preprocessing separate.
- Preserve chronological ordering for temporal splits.
- Fit imputers, encoders, scalers, feature selectors, and resampling methods using training data only.
- Clearly separate features available at transaction time from post-transaction or offline features.
- Save processed artifacts under `DATA_DIR / "processed"`.

**EDA complete. No model has been trained in this notebook.**